# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This notebook follows best practices for referencing data entities using their `@id` values as defined in the Croissant schema.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset (such as record sets, fields, and columns) are referenced by their `@id` values. Let's enumerate all available record sets and their fields for exploration:

In [ ]:
# Get all record sets by @id
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '<no name>')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            fname = field.get('name', '<no name>')
            print(f"    @id: {field['@id']} | name: {fname}")
    print()
# Optionally, print columns if available
    if 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            cname = col.get('name', '<no name>')
            print(f"    @id: {col['@id']} | name: {cname}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we'll dynamically extract data from all available record sets (using their `@id`) and load them into pandas DataFrames for further processing. Choose appropriate record sets and field `@id`s for your analysis.

In [ ]:
# Define list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record Set @id: {record_set_id} | DataFrame shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head(2), '\n')
    else:
        print(f"Record Set @id: {record_set_id} has no data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select one record set for analysis and apply operations on relevant numeric fields. For demonstration, we'll select the main survey data record set.

In [ ]:
# Choose the main survey record set (@id from overview):
main_rs_id = None
for rs in dataset.metadata.record_sets:
    if 'survey' in (rs.get('name','').lower()):
        main_rs_id = rs['@id']
        break
# Fallback: use the first record set
if main_rs_id is None:
    main_rs_id = record_set_ids[0]

df = dataframes.get(main_rs_id)
if df is None:
    print(f"Record set {main_rs_id} has no records loaded.")
else:
    # Identify numeric fields by @id
    fields = [f for f in dataset.metadata.record_sets[record_set_ids.index(main_rs_id)]['fields'] if f.get('dataType') in ['schema:Integer','schema:Float','schema:Number']]
    if fields:
        numeric_field_id = fields[0]['@id']
        numeric_field_name = fields[0]['name']
    else:
        # Fallback: take a likely numeric column
        possible_numeric = [col for col in df.columns if ('score' in col.lower() or 'age' in col.lower())]
        if possible_numeric:
            numeric_field_id = possible_numeric[0]
            numeric_field_name = numeric_field_id
        else:
            print("No numeric field found.")
            numeric_field_id = df.columns[0]
            numeric_field_name = numeric_field_id

    # Filtering and normalizing
    threshold = df[numeric_field_name].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_name]) else 10
    filtered_df = df[df[numeric_field_name] > threshold]
    print(f"Filtered records with {numeric_field_name} (field @id: {numeric_field_id}) > {threshold}:")
    print(filtered_df.head(3))

    # Normalize numeric field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_name]):
        norm_col = f"{numeric_field_name}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()
        print(f"Normalized {numeric_field_name} for filtered records:")
        print(filtered_df[[numeric_field_name, norm_col]].head())

    # Grouping (e.g., by education or village):
    group_candidates = [col for col in df.columns if ('education' in col.lower() or 'village' in col.lower())]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization examples
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    # Histogram of normalized numeric field
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[norm_col], bins=20, kde=True)
        plt.title(f"Distribution of Normalized {numeric_field_name} (field @id: {numeric_field_id})")
        plt.xlabel(norm_col)
        plt.ylabel('Count')
        plt.show()

    # Boxplot by group field
    if group_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_name])
        plt.title(f"{numeric_field_name} by {group_field} (grouped by @id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_name)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and exploratory analysis of the FAIR² Mental Health Survey Dataset using `mlcroissant`.

- Entities and fields are consistently referenced by their `@id` as per Croissant schema best practices.
- Data was dynamically loaded and analyzed from the dataset's record sets, facilitating reproducible and transparent workflows.
- Exploratory analysis and visualizations highlighted distributions and group differences in quantitative fields, providing an initial understanding of the dataset.

Further analyses can be performed by referencing additional record sets and fields via their `@id` values.